In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
PPS Design Generator

This script generates experimental design files for a Peripersonal Space (PPS) experiment.
It creates CSV files with precise timing information for audio stimuli, using regular
8-second intervals for box breathing cycles.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 4,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'box_breathing': {
        'cycle_duration_sec': 8,      # Total duration of one breath cycle in seconds
        'intro_duration_sec': 60,     # Initial settling period before trials begin
        'outro_duration_sec': 30,     # Closing period after trials end
        'alternating_phases': True,   # Alternate between inhale and exhale phases
        'sample_rate': 48000,         # Audio sample rate
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def generate_breath_timestamps(self):
        """
        Generate timestamps for breath cycles at regular 8-second intervals.
        Each timestamp represents a point where a stimulus could be inserted.
        IMPORTANT: The breath phase (inhale/exhale) strictly alternates as this
        is a fixed physiological cycle that cannot be reordered.
        
        Returns:
            DataFrame with columns: timestamp, milliseconds, sample_index, breathphase
        """
        # Calculate parameters
        total_trials = sum(self.trial_counts.values())
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        intro_sec = self.config['box_breathing']['intro_duration_sec']
        outro_sec = self.config['box_breathing']['outro_duration_sec']
        sample_rate = self.config['box_breathing']['sample_rate']
        
        # We need enough timestamps for all trials with intro/outro period
        total_duration_sec = intro_sec + (total_trials * cycle_sec) + outro_sec
        
        # Generate timestamps at regular intervals
        timestamps = []
        
        # Start with intro period
        current_time_sec = intro_sec  # First trial starts after intro
        
        # Always start with inhale (physiologically correct)
        # Strict alternation: inhale, exhale, inhale, exhale...
        for i in range(total_trials):
            # Convert time to minutes:seconds format
            minutes = int(current_time_sec / 60)
            seconds = current_time_sec % 60
            timestamp = f"{minutes:02}:{seconds:.1f}"
            
            # Calculate milliseconds and sample index
            milliseconds = current_time_sec * 1000
            sample_index = int(current_time_sec * sample_rate)
            
            # Strict alternation of breath phases - this cannot be randomized
            # In box breathing, phases must follow the natural breath cycle
            breathphase = 'inhale' if i % 2 == 0 else 'exhale'
            
            # Add this timestamp
            timestamps.append({
                'timestamp': timestamp,
                'milliseconds': milliseconds,
                'sample_index': sample_index,
                'breathphase': breathphase,
                'trial_index': i
            })
            
            # Advance to next cycle
            current_time_sec += cycle_sec
        
        # Convert to DataFrame
        timestamps_df = pd.DataFrame(timestamps)
        
        if self.config['debug_mode']:
            print(f"Generated {len(timestamps_df)} breath timestamps")
            print(f"First timestamp: {timestamps_df['timestamp'].iloc[0]}")
            print(f"Last timestamp: {timestamps_df['timestamp'].iloc[-1]}")
            print(f"Total audio duration: {total_duration_sec} seconds")
            
            # Check breath phase alternation
            phases = timestamps_df['breathphase'].tolist()
            print(f"Breath phases: {phases[:10]}... (alternating inhale-exhale pattern)")
        
        return timestamps_df
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for one participant.
        """
        # Set seed based on participant ID for reproducibility
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build the main conditions (inhale, exhale, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combos of (noise type × SOA)
            combo_list = []
            
            # Generate balanced combinations for each stimulus type and SOA
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            
            # Shuffle the combinations
            random.shuffle(combo_list)
            
            # Create trial records
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            
            # Ensure each stimulus type and SOA appears equally in catch trials
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            
            while len(catch_combos) < catch_count:
                # Find stimulus type with fewest occurrences
                min_stim = min(stim_counts, key=stim_counts.get)
                # Find SOA with fewest occurrences
                min_soa = min(soa_counts, key=soa_counts.get)
                
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            
            # Create catch trial entries
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        # Convert to DataFrame and shuffle
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        
        # Add a trial_number
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        # Check distributions for proper counterbalancing
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no looming stimuli
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no SOA relationship
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign trial types to fixed breath phases (inhale/exhale cycles).
        
        IMPORTANT: Unlike the previous implementation, we don't choose which breath
        phase to assign a trial to - rather, we choose which trial type goes into
        each predefined breath phase slot. The breath phases are fixed in their
        alternating pattern.
        """
        # Generate timestamps with fixed alternating breath phases
        timestamps_df = self.generate_breath_timestamps()
        
        # Split timestamp dataframe by breath phase
        inhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'inhale'].copy()
        exhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'exhale'].copy()
        
        # Split trials by preferred breath phase
        inhalation_trials = design_df[design_df['trial_type'] == 'inhalation'].copy()
        exhalation_trials = design_df[design_df['trial_type'] == 'exhalation'].copy()
        other_trials = design_df[~design_df['trial_type'].isin(['inhalation', 'exhalation'])].copy()
        
        # Shuffle the trials to randomize their order
        inhalation_trials = inhalation_trials.sample(frac=1).reset_index(drop=True)
        exhalation_trials = exhalation_trials.sample(frac=1).reset_index(drop=True)
        other_trials = other_trials.sample(frac=1).reset_index(drop=True)
        
        # Check if we have enough timestamps of each type
        if len(inhale_timestamps) < len(inhalation_trials):
            print(f"Warning: Not enough inhale phases ({len(inhale_timestamps)}) for all inhalation trials ({len(inhalation_trials)})")
        
        if len(exhale_timestamps) < len(exhalation_trials):
            print(f"Warning: Not enough exhale phases ({len(exhale_timestamps)}) for all exhalation trials ({len(exhalation_trials)})")
        
        # Prepare result dataframe
        result_rows = []
        
        # First, assign inhalation trials to inhale phases (as many as possible)
        used_inhale_indices = []
        for i, trial in enumerate(inhalation_trials.itertuples()):
            if i < len(inhale_timestamps):
                timestamp = inhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'inhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_inhale_indices.append(i)
            else:
                break  # Out of inhale phases
        
        # Then, assign exhalation trials to exhale phases (as many as possible)
        used_exhale_indices = []
        for i, trial in enumerate(exhalation_trials.itertuples()):
            if i < len(exhale_timestamps):
                timestamp = exhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'exhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_exhale_indices.append(i)
            else:
                break  # Out of exhale phases
        
        # Collect remaining trials that couldn't get their preferred breath phase
        remaining_inhalation = [row for i, row in enumerate(inhalation_trials.itertuples()) 
                               if i >= len(inhale_timestamps)]
        remaining_exhalation = [row for i, row in enumerate(exhalation_trials.itertuples()) 
                                if i >= len(exhale_timestamps)]
        all_remaining = remaining_inhalation + remaining_exhalation + list(other_trials.itertuples())
        random.shuffle(all_remaining)  # Mix them up
        
        # Assign remaining trials to any available timestamps
        unused_inhale = [timestamp for i, timestamp in enumerate(inhale_timestamps.itertuples()) 
                         if i >= len(used_inhale_indices)]
        unused_exhale = [timestamp for i, timestamp in enumerate(exhale_timestamps.itertuples()) 
                        if i >= len(used_exhale_indices)]
        
        # Combine all unused timestamps in their natural order
        all_unused = []
        for timestamp in timestamps_df.itertuples():
            is_inhale = timestamp.breathphase == 'inhale'
            inhale_idx = timestamp.trial_index // 2  # Convert to index within inhale/exhale group
            exhale_idx = timestamp.trial_index // 2
            
            if (is_inhale and inhale_idx >= len(used_inhale_indices)) or \
               (not is_inhale and exhale_idx >= len(used_exhale_indices)):
                all_unused.append(timestamp)
        
        # Sort by trial_index to maintain the natural breath cycle order
        all_unused.sort(key=lambda x: x.trial_index)
        
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            if pd.isna(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        if self.config['debug_mode']:
            # Verify the breathphase alternation is maintained
            phases = design_with_timestamps.sort_values(by='breath_cycle_index')['breathphase'].tolist()
            alternating = True
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternating = False
                    break
            
            if alternating:
                print("✓ Breath phases maintain perfect alternation (inhale-exhale-inhale-...)")
            else:
                print("⚠ WARNING: Breath phases do not alternate correctly!")
            
            # Show trial type distribution across breath phases
            print("\nTrial Type Distribution in Breath Phases:")
            pivot = pd.crosstab(design_with_timestamps['trial_type'], design_with_timestamps['breathphase'])
            print(pivot)
        
        return design_with_timestamps
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        return design_with_timestamps
    
    def finalize_design_csv(self, design_df):
        """
        Finalize the design CSV with proper column names and special case handling.
        
        Args:
            design_df: DataFrame with raw design data including timestamps
            
        Returns:
            DataFrame with finalized column names and special case handling
        """
        # Rename columns
        design_df.rename(columns={
            'stimulus_type': 'looming_stimulus_type',
            'timestamp_original': 'retentionphase_timestamp',
            'timestamp_after_jitter': 'looming_stimulus_timestamp',
            'timestamp_with_soa': 'tactile_stimulus_timestamp'
        }, inplace=True)
        
        # Handle baseline trials (no looming stimulus)
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # Handle catch trials (no tactile stimulus)
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected response flag (true for all except catch)
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        final_cols = [
            'soa_value_ms',              # time between looming and tactile
            'jitter_ms',                 # random jitter added to base timestamp
            'sample_index',              # sample index of retention phase
            'is_tactile',                # whether trial has tactile component
            'expected_response'          # whether response is expected
        ]
        
        # Keep only columns that exist in the DataFrame
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        # Validate final design
        self._validate_design(design_df)
        
        return design_df
    
    def _validate_design(self, design_df):
        """Check if the final design meets counterbalancing criteria."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Check trial type distribution
        print("Trial type counts:")
        trial_counts = design_df['trial_type'].value_counts()
        print(trial_counts)
        
        # 2. Check stimulus type distribution within each trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No looming in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            stim_counts = type_df['looming_stimulus_type'].value_counts()
            print(f"  {trial_type}: {stim_counts.to_dict()}")
        
        # 3. Check SOA distribution within each trial type
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No SOA relationship in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            soa_counts = type_df['soa_value_ms'].value_counts()
            print(f"  {trial_type}: {soa_counts.to_dict()}")
        
        # 4. Check breath phase distribution within each trial type
        print("\nBreath phase distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            type_df = design_df[design_df['trial_type'] == trial_type]
            phase_counts = type_df['breathphase'].value_counts()
            print(f"  {trial_type}: {phase_counts.to_dict()}")
        
        # 5. Check sequence properties
        trial_sequence = design_df['trial_type'].tolist()
        
        # Count runs of consecutive trial types
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                # Record the run we just finished
                length_key = f"{current_type}_{current_count}"
                consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
                # Reset for new type
                current_type = t
                current_count = 1
        
        # Record the final run
        length_key = f"{current_type}_{current_count}"
        consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
        
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 6. Calculate transition probabilities between trial types
        transitions = {}
        for i in range(len(trial_sequence) - 1):
            transition = (trial_sequence[i], trial_sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        
        print("\nTransition counts between trial types:")
        for transition, count in sorted(transitions.items()):
            print(f"  {transition[0]} → {transition[1]}: {count}")
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            
            # 1) Generate base design (counterbalanced combos)
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # 2) Assign each trial to a breath hold (inhale or exhale)
            design_df = self.assign_breath_holds(design_df)
            
            # 3) Finalize columns (rename, set baseline/catch to NA, etc.)
            design_df = self.finalize_design_csv(design_df)
            
            # 4) Save CSV
            out_path = os.path.join(self.experiment_log_dir,
                                    f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        
        Args:
            participant_ids: List of specific participant IDs to process
            num_participants: Number of participants to generate if participant_ids not provided
            
        Returns:
            Dictionary mapping participant IDs to generation results
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        
        return results

# ======================================================================
# AUDIO GENERATION FUNCTIONS
# ======================================================================

def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    
    This is a placeholder for audio generation functionality.
    The actual implementation would:
    1. Read the design CSV
    2. Create or load the base box breathing audio
    3. Insert looming and tactile stimuli at the specified timestamps
    4. Save the resulting audio files
    
    Args:
        design_csv_path: Path to the design CSV
        output_dir: Directory to save audio files
        base_audio_path: Optional path to base box breathing audio
        
    Returns:
        Tuple of (looming_audio_path, tactile_audio_path)
    """
    # This is just a placeholder - implement actual audio generation here
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ======================================================================
# MAIN FUNCTION
# ======================================================================

def main():
    """Main function to run the design generator."""
    # Create the design generator
    generator = PPSDesignGenerator(CONFIG)
    
    # Run for all participants defined in CONFIG
    results = generator.run()
    
    # Print summary
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()

Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'alternating_phases': True, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink':

Traceback (most recent call last):
  File "c:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'trial_type'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_10700\1864403422.py", line 636, in process_participant
    design_df = self.finalize_design_csv(design_df)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\